# 02 — Neural Network from Scratch

**Goal:** Implement a modular neural network framework in NumPy: layers, activations, a Sequential container, and a training loop.

---

## Key Formulas

**Linear layer:** $z = xW + b$, where $x \in \mathbb{R}^{N \times D_{in}}$, $W \in \mathbb{R}^{D_{in} \times D_{out}}$, $b \in \mathbb{R}^{1 \times D_{out}}$

**Gradients for linear layer:**
$$\frac{\partial L}{\partial W} = x^T \cdot \frac{\partial L}{\partial z}, \quad \frac{\partial L}{\partial b} = \mathbf{1}^T \cdot \frac{\partial L}{\partial z}, \quad \frac{\partial L}{\partial x} = \frac{\partial L}{\partial z} \cdot W^T$$

**Xavier init:** $W \sim \mathcal{N}(0, \frac{2}{D_{in} + D_{out}})$ — keeps variance stable through layers with tanh/sigmoid  
**He init:** $W \sim \mathcal{N}(0, \frac{2}{D_{in}})$ — accounts for ReLU killing half the neurons

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
np.random.seed(42)
%matplotlib inline

## 1. Layer Base Class

In [ ]:
class Layer:
    """Base class for all layers."""
    
    def __init__(self):
        self.params = {}   # name -> np.array
        self.grads = {}    # name -> np.array (same keys as params)
        self.cache = {}    # store forward pass values needed for backward
        self.training = True
    
    def forward(self, x):
        raise NotImplementedError
    
    def backward(self, dout):
        raise NotImplementedError
    
    def __call__(self, x):
        return self.forward(x)
    
    def parameters(self):
        """Return list of (param_array, grad_array) tuples."""
        return [(self.params[k], self.grads[k]) for k in self.params]

print("Layer base class defined.")

## 2. Linear (Dense) Layer

In [ ]:
class Linear(Layer):
    """Fully connected layer: z = xW + b"""
    
    def __init__(self, in_features, out_features, init='he', bias=True):
        super().__init__()
        # Weight initialization
        if init == 'he':
            std = np.sqrt(2.0 / in_features)
        elif init == 'xavier':
            std = np.sqrt(2.0 / (in_features + out_features))
        elif init == 'normal':
            std = 0.01
        else:
            std = np.sqrt(1.0 / in_features)
        
        self.params['W'] = np.random.randn(in_features, out_features) * std
        self.grads['W'] = np.zeros_like(self.params['W'])
        
        self.use_bias = bias
        if bias:
            self.params['b'] = np.zeros((1, out_features))
            self.grads['b'] = np.zeros_like(self.params['b'])
    
    def forward(self, x):
        self.cache['x'] = x
        out = x @ self.params['W']
        if self.use_bias:
            out = out + self.params['b']
        return out
    
    def backward(self, dout):
        x = self.cache['x']
        self.grads['W'] = x.T @ dout                    # (D_in, N) @ (N, D_out)
        if self.use_bias:
            self.grads['b'] = np.sum(dout, axis=0, keepdims=True)  # (1, D_out)
        dx = dout @ self.params['W'].T                   # (N, D_out) @ (D_out, D_in)
        return dx

# Quick test
layer = Linear(3, 2)
x = np.random.randn(5, 3)  # batch of 5, 3 features
out = layer(x)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")
print(f"W shape: {layer.params['W'].shape}, b shape: {layer.params['b'].shape}")

## 3. Activation Layers

In [ ]:
class ReLU(Layer):
    def forward(self, x):
        self.cache['mask'] = (x > 0).astype(float)
        return x * self.cache['mask']
    
    def backward(self, dout):
        return dout * self.cache['mask']


class Tanh(Layer):
    def forward(self, x):
        self.cache['out'] = np.tanh(x)
        return self.cache['out']
    
    def backward(self, dout):
        return dout * (1.0 - self.cache['out'] ** 2)


class Sigmoid(Layer):
    def forward(self, x):
        self.cache['out'] = 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))
        return self.cache['out']
    
    def backward(self, dout):
        s = self.cache['out']
        return dout * s * (1.0 - s)


print("Activation layers defined.")

## 4. Loss Functions

In [ ]:
class MSELoss:
    def __call__(self, pred, target):
        self.pred = pred
        self.target = target
        self.N = pred.shape[0]
        return np.mean((pred - target) ** 2)
    
    def backward(self):
        return 2.0 * (self.pred - self.target) / self.N


class CrossEntropyLoss:
    """Softmax + Cross-Entropy combined for numerical stability."""
    def __call__(self, logits, targets):
        """logits: (N, C), targets: (N,) integer class labels."""
        self.N = logits.shape[0]
        # Stable softmax
        shifted = logits - np.max(logits, axis=1, keepdims=True)
        exp_scores = np.exp(shifted)
        self.probs = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)
        self.targets = targets
        # NLL
        log_probs = -np.log(self.probs[np.arange(self.N), targets] + 1e-12)
        return np.mean(log_probs)
    
    def backward(self):
        dlogits = self.probs.copy()
        dlogits[np.arange(self.N), self.targets] -= 1.0
        return dlogits / self.N


class BCELoss:
    """Binary cross-entropy (expects sigmoid outputs)."""
    def __call__(self, pred, target):
        self.pred = np.clip(pred, 1e-7, 1 - 1e-7)
        self.target = target
        self.N = pred.shape[0]
        return -np.mean(target * np.log(self.pred) + (1 - target) * np.log(1 - self.pred))
    
    def backward(self):
        return (-(self.target / self.pred) + (1 - self.target) / (1 - self.pred)) / self.N


print("Loss functions defined.")

## 5. Sequential Model Container

In [ ]:
class Sequential:
    """Chain layers sequentially: forward passes through in order, backward in reverse."""
    
    def __init__(self, layers):
        self.layers = layers
    
    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    
    def __call__(self, x):
        return self.forward(x)
    
    def backward(self, dout):
        for layer in reversed(self.layers):
            dout = layer.backward(dout)
        return dout
    
    def parameters(self):
        """Collect all parameters from all layers."""
        all_params = []
        for layer in self.layers:
            all_params.extend(layer.parameters())
        return all_params
    
    def train(self):
        for layer in self.layers:
            layer.training = True
    
    def eval(self):
        for layer in self.layers:
            layer.training = False


print("Sequential model defined.")

## 6. Optimizer

In [ ]:
class SGD:
    def __init__(self, parameters, lr=0.01, momentum=0.0):
        self.parameters = parameters  # list of (param, grad) tuples
        self.lr = lr
        self.momentum = momentum
        self.velocities = [np.zeros_like(p) for p, _ in parameters]
    
    def step(self):
        for i, (param, grad) in enumerate(self.parameters):
            self.velocities[i] = self.momentum * self.velocities[i] - self.lr * grad
            param += self.velocities[i]   # in-place update
    
    def zero_grad(self):
        for _, grad in self.parameters:
            grad[:] = 0


print("SGD optimizer defined.")

## 7. Demo 1 — XOR Problem

XOR is the classic non-linearly-separable problem. A single-layer perceptron cannot solve it; we need at least one hidden layer.

In [ ]:
np.random.seed(42)

# XOR data
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=np.float64)
Y = np.array([[0], [1], [1], [0]], dtype=np.float64)

# 2-layer network: 2 -> 8 -> 1
model = Sequential([
    Linear(2, 8, init='he'),
    Tanh(),
    Linear(8, 1, init='he'),
    Sigmoid()
])

loss_fn = MSELoss()
optimizer = SGD(model.parameters(), lr=1.0, momentum=0.9)

losses = []
for epoch in range(1000):
    # Forward
    pred = model(X)
    loss = loss_fn(pred, Y)
    losses.append(loss)
    
    # Backward
    optimizer.zero_grad()
    dout = loss_fn.backward()
    model.backward(dout)
    
    # Update
    optimizer.step()

# Results
print("XOR predictions:")
pred = model(X)
for i in range(4):
    print(f"  {X[i]} => {pred[i,0]:.4f} (target: {Y[i,0]})")

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(losses)
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
ax.set_title('XOR Training')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Demo 2 — Synthetic Classification with Decision Boundaries

Generate a 2D spiral dataset (3 classes) and train a network on it.

In [ ]:
def make_spirals(n_samples=300, n_classes=3, noise=0.2):
    X = np.zeros((n_samples * n_classes, 2))
    Y = np.zeros(n_samples * n_classes, dtype=int)
    for c in range(n_classes):
        idx = range(n_samples * c, n_samples * (c + 1))
        r = np.linspace(0.0, 1.0, n_samples)
        t = np.linspace(c * 4, (c + 1) * 4, n_samples) + np.random.randn(n_samples) * noise
        X[idx] = np.c_[r * np.sin(t), r * np.cos(t)]
        Y[idx] = c
    return X, Y

np.random.seed(42)
X_train, Y_train = make_spirals()

fig, ax = plt.subplots(figsize=(5, 5))
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
for c in range(3):
    mask = Y_train == c
    ax.scatter(X_train[mask, 0], X_train[mask, 1], c=colors[c], s=10, alpha=0.7, label=f'Class {c}')
ax.set_title('Spiral Dataset (3 classes)')
ax.legend()
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

In [ ]:
# Train a deeper network on spirals
np.random.seed(42)

model = Sequential([
    Linear(2, 64, init='he'),
    ReLU(),
    Linear(64, 64, init='he'),
    ReLU(),
    Linear(64, 3, init='he'),  # 3 classes, logits (softmax in loss)
])

loss_fn = CrossEntropyLoss()
optimizer = SGD(model.parameters(), lr=0.5, momentum=0.9)

N = X_train.shape[0]
batch_size = 64
losses = []
accs = []

for epoch in range(200):
    # Shuffle
    perm = np.random.permutation(N)
    X_shuf = X_train[perm]
    Y_shuf = Y_train[perm]
    
    epoch_loss = 0
    for i in range(0, N, batch_size):
        xb = X_shuf[i:i+batch_size]
        yb = Y_shuf[i:i+batch_size]
        
        # Forward
        logits = model(xb)
        loss = loss_fn(logits, yb)
        epoch_loss += loss * xb.shape[0]
        
        # Backward
        optimizer.zero_grad()
        dout = loss_fn.backward()
        model.backward(dout)
        optimizer.step()
    
    losses.append(epoch_loss / N)
    # Accuracy
    logits = model(X_train)
    preds = np.argmax(logits, axis=1)
    acc = np.mean(preds == Y_train)
    accs.append(acc)
    
    if epoch % 40 == 0:
        print(f"Epoch {epoch:3d}: loss={losses[-1]:.4f}, acc={acc:.3f}")

print(f"Final accuracy: {accs[-1]:.3f}")

In [ ]:
# Plot decision boundary
def plot_decision_boundary(model, X, Y, title='Decision Boundary'):
    h = 0.02
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    grid = np.c_[xx.ravel(), yy.ravel()]
    
    logits = model(grid)
    Z = np.argmax(logits, axis=1).reshape(xx.shape)
    
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=ListedColormap(['#FF6B6B', '#4ECDC4', '#45B7D1']))
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
    for c in range(3):
        mask = Y == c
        ax.scatter(X[mask, 0], X[mask, 1], c=colors[c], s=10, edgecolors='k', linewidth=0.3)
    ax.set_title(title)
    ax.set_aspect('equal')
    plt.tight_layout()
    plt.show()

plot_decision_boundary(model, X_train, Y_train, 'Spiral Classification — Decision Boundary')

In [ ]:
# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
ax1.plot(losses)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Training Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(accs)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.set_title('Training Accuracy')
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Feature Space Transformation — Layer by Layer

Visualize how each layer transforms the input space. The network learns to "untangle" the spiral.

In [ ]:
# Extract intermediate representations
activations = [X_train]  # input
labels_for_plot = ['Input']
x = X_train.copy()

for i, layer in enumerate(model.layers):
    x = layer(x)
    if isinstance(layer, (Linear, ReLU)):  # Skip storing after every layer to keep it clean
        activations.append(x.copy())
        name = f"After {type(layer).__name__} ({x.shape[1]}D)" if isinstance(layer, Linear) else f"After ReLU"
        labels_for_plot.append(name)

# Use PCA to project high-dim activations to 2D for visualization
from numpy.linalg import svd

def pca_2d(X):
    X_centered = X - X.mean(axis=0)
    if X.shape[1] <= 2:
        return X[:, :2]
    U, S, Vt = svd(X_centered, full_matrices=False)
    return X_centered @ Vt[:2].T

fig, axes = plt.subplots(1, len(activations), figsize=(4 * len(activations), 4))
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

for idx, (act, label) in enumerate(zip(activations, labels_for_plot)):
    proj = pca_2d(act)
    ax = axes[idx]
    for c in range(3):
        mask = Y_train == c
        ax.scatter(proj[mask, 0], proj[mask, 1], c=colors[c], s=5, alpha=0.5)
    ax.set_title(label, fontsize=9)
    ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('How the Network Transforms Feature Space (PCA projections)', y=1.02)
plt.tight_layout()
plt.show()

## 10. Xavier vs He Initialization — Why It Matters

In [ ]:
# Show variance collapse with bad initialization
def forward_and_track_variance(init_type, depth=10, width=256, activation='relu'):
    """Track activation variance through a deep network."""
    np.random.seed(42)
    x = np.random.randn(100, width)
    variances = [np.var(x)]
    
    for _ in range(depth):
        if init_type == 'he':
            W = np.random.randn(width, width) * np.sqrt(2.0 / width)
        elif init_type == 'xavier':
            W = np.random.randn(width, width) * np.sqrt(2.0 / (width + width))
        elif init_type == 'small':
            W = np.random.randn(width, width) * 0.01
        elif init_type == 'large':
            W = np.random.randn(width, width) * 1.0
        
        x = x @ W
        if activation == 'relu':
            x = np.maximum(0, x)
        elif activation == 'tanh':
            x = np.tanh(x)
        variances.append(np.var(x))
    
    return variances

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# With ReLU
for init in ['he', 'xavier', 'small', 'large']:
    v = forward_and_track_variance(init, activation='relu')
    ax1.plot(v, label=init, linewidth=2)
ax1.set_yscale('log')
ax1.set_xlabel('Layer'); ax1.set_ylabel('Activation variance (log)')
ax1.set_title('Activation Variance with ReLU')
ax1.legend(); ax1.grid(True, alpha=0.3)

# With Tanh
for init in ['he', 'xavier', 'small', 'large']:
    v = forward_and_track_variance(init, activation='tanh')
    ax2.plot(v, label=init, linewidth=2)
ax2.set_yscale('log')
ax2.set_xlabel('Layer'); ax2.set_ylabel('Activation variance (log)')
ax2.set_title('Activation Variance with Tanh')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("He init keeps variance stable with ReLU; Xavier is better suited for tanh/sigmoid.")

## 11. Mini-Batch Training Utility

In [ ]:
def train(model, X, Y, loss_fn, optimizer, epochs=100, batch_size=32, verbose=True):
    """General-purpose training loop."""
    N = X.shape[0]
    history = {'loss': [], 'acc': []}
    
    for epoch in range(epochs):
        perm = np.random.permutation(N)
        epoch_loss = 0.0
        
        for i in range(0, N, batch_size):
            xb = X[perm[i:i+batch_size]]
            yb = Y[perm[i:i+batch_size]]
            
            pred = model(xb)
            loss = loss_fn(pred, yb)
            epoch_loss += loss * xb.shape[0]
            
            optimizer.zero_grad()
            dout = loss_fn.backward()
            model.backward(dout)
            optimizer.step()
        
        history['loss'].append(epoch_loss / N)
        
        # Accuracy (for classification)
        logits = model(X)
        if logits.shape[1] > 1:
            preds = np.argmax(logits, axis=1)
            acc = np.mean(preds == Y)
        else:
            preds = (logits > 0.5).astype(int).ravel()
            acc = np.mean(preds == Y.ravel())
        history['acc'].append(acc)
        
        if verbose and epoch % (epochs // 5) == 0:
            print(f"Epoch {epoch:4d}: loss={history['loss'][-1]:.4f}, acc={acc:.3f}")
    
    return history

print("Training utility defined.")

## Interview Quick Reference

**Q: Walk me through a forward and backward pass.**  
Forward: input flows through layers (linear transform + activation) to produce output/loss. Backward: starting from loss gradient, each layer computes gradient w.r.t. its inputs and parameters using chain rule, passing input gradient to previous layer.

**Q: Why can't a single-layer perceptron solve XOR?**  
XOR is not linearly separable. A single linear boundary cannot separate the four XOR points. Need at least one hidden layer to create a non-linear decision boundary.

**Q: He vs Xavier initialization?**  
He: $\text{Var}(W) = 2/n_{in}$ — designed for ReLU (which kills half the variance). Xavier: $\text{Var}(W) = 2/(n_{in} + n_{out})$ — for sigmoid/tanh. Wrong init leads to vanishing/exploding activations in deep nets.

**Q: Why mini-batch gradient descent?**  
- Full batch: accurate gradients but slow per update, needs all data in memory
- SGD (batch=1): noisy, fast updates, bad GPU utilization
- Mini-batch: sweet spot — enough noise for regularization, good GPU utilization, stable convergence

---
*Next: 03_activations_and_losses.ipynb — deep dive into activation functions and loss functions*